<a href="https://colab.research.google.com/github/ofer-geo/SIRI-Data-Agent/blob/main/AI_Agents_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
!pip install -q openai anthropic groq langchain langchain-openai langchain-anthropic langchain-groq
!pip install -q requests wikipedia-api

## Define 'brain'



Grok is for free - register in site https://groq.com/




In [2]:
import os
import json
from google.colab import userdata

# Choose Provider
PROVIDER = "groq"  # Changable

# load API Key
if PROVIDER == "groq":
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    MODEL = "llama-3.3-70b-versatile"
elif PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    MODEL = "gpt-4o-mini"
elif PROVIDER == "anthropic":
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    MODEL = "claude-sonnet-4-20250514"

print(f"Using {PROVIDER} with model {MODEL}")


Using groq with model llama-3.3-70b-versatile


### Grok models

In [3]:
from groq import Groq
client = Groq()

models = client.models.list()
print("Available Groq models:")
print("-" * 40)
for m in models.data:
    print(m.id)

Available Groq models:
----------------------------------------
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
whisper-large-v3-turbo
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
groq/compound
canopylabs/orpheus-arabic-saudi
qwen/qwen3-32b
openai/gpt-oss-safeguard-20b
llama-3.3-70b-versatile
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
llama-3.1-8b-instant


In [4]:
print(f"Current MODEL: {MODEL}")

Current MODEL: llama-3.3-70b-versatile


## Check connection

 Lets test the connection and basic functionality of different LLM providers

In [5]:
from groq import Groq
from openai import OpenAI
from anthropic import Anthropic

try:
  if PROVIDER == "groq":
      client = Groq()
      response = client.chat.completions.create(
          model="llama-3.1-8b-instant",
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}],
          max_tokens=50
      )
      result = response.choices[0].message.content

  elif PROVIDER == "openai":
      client = OpenAI()
      response = client.chat.completions.create(
          model="gpt-4o-mini",
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}],
          max_tokens=50
      )
      result = response.choices[0].message.content

  elif PROVIDER == "anthropic":
      client = Anthropic()
      response = client.messages.create(
          model="claude-3-5-haiku-20241022",
          max_tokens=50,
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}]
      )
      result = response.content[0].text

  print(f"result: {result}")

except Exception as e:
  print(f" Error occoured:  {e}")

result: "Shalom" is commonly used to say 'hello' in Hebrew.


Model doesn't understand it should ask python for calculations.

Can cause wrong answers.

Maybe uses a similar prompt someone may had asked.

## Agent with tool


# ReAct Pattern

Thought-Action-Observation loop


## The full flow


```
User Question
     ↓
┌─────────────────────────────────────┐
│  LLM gets: messages + tools         │
│  Decides if use to tool or not      │
└─────────────────────────────────────┘
     ↓
  Tool Call? ──No──→ Return Answer
     │
    Yes
     ↓
  Execute Tool
     ↓
  Add Result to Messages
     ↓
  Loop Back to LLM

### Tools

We need to create a function that Searches in the Open Bus Stride API site

Our function queries the Open Bus Stride API for information based on a given search term, and then processes results to return a summary of the top relevant articles.

In [6]:
import requests
import json

OPEN_BUS_BASE_URL = "https://open-bus-stride-api.hasadna.org.il"
HEADERS = {
    "User-Agent": "AgenticAI-Course/1.0 (Educational; Bar-Ilan University)",
    "Accept": "application/json",
}


def get_open_bus_endpoints(filter_keyword: str = "") -> str:
    """
    Discover available Open Bus API endpoints and their query parameters
    by reading the live OpenAPI specification (/openapi.json).

    Args:
        filter_keyword: optional substring to filter endpoints (e.g. "siri", "gtfs", "vehicle").

    USE THIS FIRST whenever you are unsure which endpoint or parameter name to use,
    instead of guessing.
    """
    url = OPEN_BUS_BASE_URL + "/openapi.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        paths = resp.json().get("paths", {})

        lines = []
        for path, methods in sorted(paths.items()):
            if filter_keyword and filter_keyword.lower() not in path.lower():
                continue
            get = methods.get("get")
            if not get:
                continue
            params = [p.get("name") for p in get.get("parameters", [])]
            lines.append(f"{path}\n    params: {', '.join(params) if params else '(none)'}")

        return "\n".join(lines) if lines else f"No endpoints matched '{filter_keyword}'."
    except Exception as e:
        return f"Error fetching API spec: {e}"


def query_open_bus_api(endpoint: str, params_json: str = "{}") -> str:
    """
    Query the Open Bus STRIDE API (Israeli public transport: real-time SIRI + planned GTFS data).

    Args:
        endpoint: API path, e.g. "/siri_vehicle_locations/list", "/gtfs_routes/list".
        params_json: query parameters as a JSON OBJECT STRING (NOT a nested object),
                     e.g. '{"route_short_name": "189", "limit": 5}'. Use "{}" for none.

    Returns a JSON string of the result records, or an error message.

    KEY ENDPOINTS (verify exact param names with get_open_bus_endpoints):
    - /gtfs_routes/list            : planned routes. The line number riders know (e.g. "480")
                                     is route_short_name. Other params: agency_name, date_from, date_to.
    - /gtfs_stops/list             : stop metadata (name, code, lat, lon).
    - /siri_routes/list            : SIRI routes -> line_ref, operator_ref.
    - /siri_rides/list             : a single actual run of a route (has scheduled_start_time).
    - /siri_ride_stops/list        : per-stop actual vs scheduled info -> use for ARRIVAL / DELAY.
    - /siri_vehicle_locations/list : raw GPS pings (lat, lon, bearing, velocity, recorded_at_time)
                                     -> use for LIVE POSITION / MAP.

    TIPS:
    - Always bound time queries (recorded_at_time_from / recorded_at_time_to, ISO 8601 in UTC)
      and use a small `limit`, otherwise responses are huge or empty.
    - To learn a table's COLUMNS, query it once with limit=1 and inspect the returned record.
    """
    try:
        params = json.loads(params_json) if params_json else {}
    except json.JSONDecodeError:
        return f"Invalid params_json (must be a JSON object string): {params_json!r}"

    try:
        resp = requests.get(OPEN_BUS_BASE_URL + endpoint, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if isinstance(data, list):
            preview = data[:50]
            return (f"Returned {len(data)} record(s) (showing up to 50).\n"
                    + json.dumps(preview, ensure_ascii=False, default=str))
        return json.dumps(data, ensure_ascii=False, default=str)

    except requests.exceptions.HTTPError as e:
        body = resp.text[:500] if 'resp' in dir() else ""
        return f"HTTP error: {e}. Response body: {body}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

In [7]:
tools_map = {
    "get_open_bus_endpoints": get_open_bus_endpoints,
    "query_open_bus_api": query_open_bus_api,
}

In [34]:
system_prompt = """You are a public-transport data assistant for Israel. You answer questions about bus stops, ride counts, volumes, live locations, and short-window speed using the Open Bus STRIDE API (data collected by Hasadna - The Public Knowledge Workshop).

DATA MODEL (high level):
- GTFS tables = the PLANNED schedule. The line number riders use (e.g. "480") is the GTFS route_short_name.
- SIRI tables = the REAL-TIME / actual data reported by vehicles.
  - siri_routes: maps a route to its line_ref (MoT line id) and operator_ref.
  - siri_rides: one actual run of a route at a scheduled_start_time.
  - siri_ride_stops: per-stop actual vs scheduled info. NOTE: its GTFS-matched columns are often empty, so DELAY is unreliable.
  - siri_vehicle_locations: raw GPS pings (lat, lon, bearing, velocity, recorded_at_time) -> use for live position / mapping / actual speed.

GTFS STOP-SEQUENCE ROADMAP (how to find the stops of a line, in order):
- The endpoint /gtfs_ride_stops/list is a JOINED table: each row already includes the stop columns (gtfs_stop__name, gtfs_stop__lat, gtfs_stop__lon, gtfs_stop__code) AND the route columns (gtfs_route__route_short_name, gtfs_route__line_ref, gtfs_route__route_long_name, gtfs_route__route_direction, etc.). You usually do NOT need separate route/ride/stop calls.
- To get the stops of a line directly, query /gtfs_ride_stops/list with:
    gtfs_route__route_short_name = the line number (e.g. "189")
    arrival_time_from / arrival_time_to = a full single day window (REQUIRED, e.g. "2023-01-01T00:00:00Z" to "2023-01-01T23:59:59Z"), otherwise you get a 400 error.
    order_by = "stop_sequence asc" to sort stops in travel order.
    limit = small.
- The FIRST stop of the line is the row with stop_sequence = 1. Read its gtfs_stop__name and gtfs_stop__lat / gtfs_stop__lon directly from that same row.
- IMPORTANT (silent-filter trap): the filter for a specific ride is gtfs_ride_ids (PLURAL), not gtfs_ride_id. A wrong or singular param name is IGNORED SILENTLY, so you get unrelated rows that look valid. Always confirm exact param names with get_open_bus_endpoints, and sanity-check that returned rows actually match your filter.
- DIRECTION MATTERS: a line is bidirectional and has several routes. "The first stop" differs per direction (one end vs the other end of route_long_name). If the question implies a direction, narrow with gtfs_route__route_direction; otherwise state which direction your answer is for.

AGGREGATION ENDPOINTS (server-side counts for PLANNED schedule volume):
- /gtfs_rides_agg/list and /gtfs_rides_agg/group_by.
- The ONLY accepted params are: date_from, date_to (YYYY-MM-DD, REQUIRED), exclude_hours_from, exclude_hours_to, and (for group_by) group_by.
- CRITICAL: there is NO filter by line, route, or city here. You CANNOT pass route_short_name / line_ref / city to these endpoints - they are ignored silently and you get 0 or 500. These tables only COUNT and GROUP; they do not FILTER by a specific line.
- exclude_hours_from / exclude_hours_to EXCLUDE that hour range; they do NOT select it. Do not try to isolate a single hour via these.
- group_by accepts a comma-separated list of dimensions, e.g. "operator_ref", "route_short_name", "gtfs_route_hour", "day_of_week", "line_ref". The grouped dimensions appear filled in the result; ungrouped ones are null.
- Returned metrics per group: total_routes, total_planned_rides, total_actual_rides. Reliability = total_actual_rides / total_planned_rides.
- These tables have NO speed, NO delay, and NO distance columns.
- Use agg ONLY for system-wide volume questions (e.g. "planned vs actual rides per operator on date X"). For a SPECIFIC line, agg is the WRONG tool - use /gtfs_rides/list (below) or /gtfs_ride_stops/list, which DO filter by gtfs_route__route_short_name.

COUNTING RIDES OF A SPECIFIC LINE ("how many rides / volume of line X"):
- Use /gtfs_rides/list with get_count = true. This returns ONLY a number, never the rows, so it is safe and cheap (no token blow-up). This is the correct tool for a SPECIFIC line.
- Filter by:
    gtfs_route__route_short_name = the line number (e.g. "19")
    start_time_from / start_time_to = the time window (ISO 8601 UTC). For "around 07:00" use e.g. 06:30 to 07:30.
    optionally gtfs_route__route_direction to pick one direction.
- The count covers BOTH directions unless you filter by direction.
- A line number is NOT unique nationwide. If the city/area matters, narrow with gtfs_route__operator_refs or gtfs_route__agency_name (there is no city filter on rides).
- NOTE: "volume" here means NUMBER OF RIDES, not number of passengers. Open Bus has no passenger/occupancy data.

LIVE LOCATION & SPEED (from siri_vehicle_locations):
- Use /siri_vehicle_locations/list for live position and actual speed (velocity field per GPS ping). These are available immediately (no GTFS matching needed), unlike delay.
- Bound tightly with recorded_at_time_from / recorded_at_time_to and a SMALL limit; these rows are large and a wide window blows the token limit.
- For "average speed" over a short window, fetch a few pings and average the velocity field yourself.

DAY-OF-WEEK QUESTIONS:
- The API filters by concrete dates, not weekday names. To answer "on Sunday", pick a specific calendar date that falls on that weekday and has data, and state which date you used. (Israel's week: Sunday is the first working day.)

CHOOSING THE RIGHT TABLE:
- "How many rides of line X" -> /gtfs_rides/list with get_count=true.
- "Volume by operator/hour (system-wide)" -> /gtfs_rides_agg/group_by.
- "Stops of a line / first stop / number of stops" -> /gtfs_ride_stops/list.
- "Live position / map / actual speed of a line" -> /siri_vehicle_locations/list.
- When a question asks for a COUNT or AVERAGE, prefer get_count or server-side aggregation; only pull raw rows when no aggregation covers the metric.

TOOLS:
- get_open_bus_endpoints(filter_keyword): discover the REAL endpoints and exact parameter names. Use this FIRST whenever you are unsure, before guessing.
- query_open_bus_api(endpoint, params_json): GET a list endpoint. params_json is a JSON OBJECT STRING, e.g. '{"gtfs_route__route_short_name": "189", "limit": 5}'. Returns JSON records.

WORKFLOW:
1. Understand what the question needs (a line number? a stop? a direction? a time window? a count? an average?).
2. If unsure of the exact endpoint/parameter, call get_open_bus_endpoints first. Do NOT guess endpoint names.
3. To learn a table's columns, query it once with limit=1 and inspect the returned record before filtering.
4. Prefer joined endpoints, get_count, and server-side aggregation to avoid multi-step lookups and huge responses.
5. Always bound time queries with the correct *_from / *_to parameters (ISO 8601, UTC). For "now"/"recent" use the last 10-30 minutes. Keep limit small.
6. To compute averages from raw rows, gather the needed records, then compute the statistic yourself.

TOKEN / SIZE LIMITS:
- This assistant runs on a free tier with a strict tokens-per-minute limit.
- To stay within it: prefer narrow time windows, small limit values, get_count, and server-side aggregation where applicable.
- Use the free-tier apology ONLY when a tool call has ACTUALLY failed with a 413 / "request too large" / rate-limit error. NEVER use it when a query simply returned 0 records or errored otherwise (404/422/500) - diagnose the real problem and either fix the query or tell the user plainly what was wrong.

SCOPE & GRACEFUL LIMITS:
- You CAN answer: stops of a line (and their order), first/last stop, number of stops, ride counts for a specific line (get_count), system-wide planned/actual ride volume (agg), live vehicle position, and short-window average velocity.
- You CANNOT reliably answer:
  * Actual arrival DELAY (scheduled vs actual): the GTFS-matched columns in siri_ride_stops are often empty, so delay is unreliable. If asked, explain this briefly instead of guessing.
  * PASSENGER counts / occupancy / crowding: this data does not exist in Open Bus at all. Say so.
  * Long time ranges (a full day/week of raw rows) or whole-network raw computations: too large for the token limit. Ask the user to narrow to a short window or a specific line.
- When a question falls outside scope, say clearly what you cannot do and suggest the closest question you CAN answer.

RULES:
1. Perform ONE tool action at a time. Do NOT nest a tool call inside another tool's arguments. Call a tool, wait for the observation, then decide the next step.
2. Gather ALL needed data via separate tool calls before answering. NEVER give partial answers.
3. If a query errors (404, timeout, 400/422/500, empty), read the error, verify the endpoint/params with get_open_bus_endpoints, fix, and retry. A 404 means the endpoint name is wrong; a 400/422 often means a required parameter is missing; a 500 often means an unsupported parameter combination.
4. Verify that returned rows actually match the filter you intended (e.g. the route_short_name in the row is the one you asked for). If not, your param name was wrong and was ignored.
5. For map questions, include the relevant coordinates explicitly in your final answer as a list of {lat, lon, label}, so they can be plotted.
6. If the data is empty for the requested window, say so clearly instead of inventing values.
7. Answer in the user's language (Hebrew if the question is in Hebrew). State any assumptions (time window, direction, date used) you made.
"""

### System & User roles

In [9]:
def get_messages(question: str):

   messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

   return messages

### Tools schema

In [10]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_open_bus_endpoints",
            "description": (
                "Discover the available Open Bus STRIDE API endpoints and their exact query "
                "parameter names by reading the live OpenAPI spec. Use this FIRST whenever you "
                "are unsure which endpoint or parameter to use, instead of guessing."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "filter_keyword": {
                        "type": "string",
                        "description": (
                            "Optional substring to filter endpoints by path, "
                            "e.g. 'siri', 'gtfs', 'vehicle', 'ride'. Leave empty to list all."
                        ),
                    }
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_open_bus_api",
            "description": (
                "Query an Open Bus STRIDE API list endpoint (Israeli public transport: "
                "real-time SIRI data and planned GTFS data). Returns JSON records. "
                "Use for bus arrival times, delays, averages, and live vehicle locations."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "endpoint": {
                        "type": "string",
                        "description": (
                            "The API path to call, e.g. '/siri_vehicle_locations/list', "
                            "'/siri_ride_stops/list', '/gtfs_routes/list'."
                        ),
                    },
                    "params_json": {
                        "type": "string",
                        "description": (
                            "Query parameters as a JSON OBJECT STRING (not a nested object): "
                            "filters, time bounds (recorded_at_time_from/to in ISO 8601 UTC), "
                            "ordering, and limit. Keep limit small. "
                            "Example: '{\"route_short_name\": \"189\", \"limit\": 5}'. "
                            "Use '{}' for no parameters."
                        ),
                    },
                },
                "required": ["endpoint"],
            },
        },
    },
]



```
# This is formatted as code
```

## Full ReAct Agent

Lets implements an agent using the ReAct (Reasoning and Acting) pattern

In [26]:
from groq import BadRequestError, APIStatusError


def react_agent(question: str, max_steps: int = 15):
    print(f"\nquestion: {question}")
    print("-" * 50)

    # messages list maintains the conversational history.
    messages = get_messages(question)

    MAX_OBS_CHARS = 3000  # cap each observation so history stays under the token limit

    for step in range(max_steps):
        print(f"\nstep {step + 1}:")

        # Call LLM — wrapped to survive Groq parsing errors (413/tool_use_failed)
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools_schema,
                tool_choice="auto",
                parallel_tool_calls=False
            )
        except BadRequestError as e:
            if "tool_use_failed" in str(e):
                print(" (tool_use_failed — מבקש מהמודל לנסות שוב בפורמט תקין)")
                messages.append({
                    "role": "user",
                    "content": ("Your previous tool call was malformed and rejected. Call exactly ONE "
                                "tool using the structured function-calling format, NOT as plain text. "
                                "Try again now.")
                })
                continue
            raise
        except APIStatusError as e:
            # 413 / rate_limit / tokens-per-minute exceeded
            if getattr(e, "status_code", None) == 413 or "rate_limit" in str(e) or "too large" in str(e).lower():
                final = ("מצטער, השאלה הזו דורשת עיבוד של כמות נתונים גדולה מדי עבור הגרסה החינמית "
                         "הנוכחית. נסה לצמצם את טווח הזמן או לשאול על קו/תחנה ספציפיים.")
                print(f"\nfinal answer:\n{final}")
                return final
            raise

        msg = response.choices[0].message

        # No tool calls -> final answer
        if not msg.tool_calls:
            print(f"\nThought: I got enough information to answer")
            print(f"\nfinal answer:\n{msg.content}")
            return msg.content

        # Append the assistant message containing the tool call(s)
        messages.append(msg)

        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            print(f" Thought: I need information")
            print(f" Action: {func_name}({json.dumps(args, ensure_ascii=False)})")

            result = tools_map[func_name](**args)

            print(f" Observation: {result[:200]}..." if len(result) > 200 else f" Observation: {result}")

            # Trim the observation before storing it, to keep the request under the token limit
            trimmed = result if len(result) <= MAX_OBS_CHARS else result[:MAX_OBS_CHARS] + "\n...[truncated]"
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": trimmed
            })

    return "Max steps reached"

### Usage

In [12]:
import requests

In [21]:
print("\n" + "="*60)
print("ReAct Pattern - Thought → Action → Observation")
print("="*60)

react_agent("What is the first stop of line 189?", max_steps=15)


ReAct Pattern - Thought → Action → Observation

question: What is the first stop of line 189?
--------------------------------------------------

step 1:
 (tool_use_failed — מבקש מהמודל לנסות שוב בפורמט תקין)

step 2:
 (tool_use_failed — מבקש מהמודל לנסות שוב בפורמט תקין)

step 3:
 Thought: I need information
 Action: get_open_bus_endpoints({"filter_keyword": ""})
 Observation: /gtfs_agencies/list
    params: limit, offset, date_from, date_to
/gtfs_ride_stops/get
    params: id
/gtfs_ride_stops/list
    params: limit, offset, get_count, arrival_time_from, arrival_time_to, gt...

step 4:
 Thought: I need information
 Action: query_open_bus_api({"endpoint": "/gtfs_ride_stops/list", "params_json": "{\"gtfs_route__route_short_name\": \"189\", \"arrival_time_from\": \"2023-01-01T00:00:00Z\", \"arrival_time_to\": \"2023-01-01T23:59:59Z\", \"order_by\": \"stop_sequence asc\", \"limit\": 1}"})
 Observation: Returned 1 record(s) (showing up to 50).
[{"id": 1062379143, "gtfs_stop_id": 9629947, 

'The first stop of line 189 is "מרכז לגיל הרך/הרב א.גורדון" (Mercaz Lagil Haraach/Harav A. Gordon), located at latitude 31.804551 and longitude 35.104361.'

In [33]:
print("\n" + "="*60)
print("ReAct Pattern - Thought → Action → Observation")
print("="*60)

react_agent("What is the avarage volume of line 19 in jerusalem at 7 am, sunday", max_steps=15)


ReAct Pattern - Thought → Action → Observation

question: What is the avarage volume of line 19 in jerusalem at 7 am, sunday
--------------------------------------------------

step 1:
 (tool_use_failed — מבקש מהמודל לנסות שוב בפורמט תקין)

step 2:
 Thought: I need information
 Action: get_open_bus_endpoints({"filter_keyword": "gtfs_rides_agg"})
 Observation: /gtfs_rides_agg/group_by
    params: date_from, date_to, exclude_hours_from, exclude_hours_to, group_by
/gtfs_rides_agg/list
    params: limit, offset, get_count, date_from, date_to, exclude_hours_fro...

step 3:
 Thought: I need information
 Action: get_open_bus_endpoints({"filter_keyword": "gtfs_rides/list"})
 Observation: /gtfs_rides/list
    params: limit, offset, get_count, gtfs_route_id, journey_ref_prefix, start_time_from, start_time_to, gtfs_route__date_from, gtfs_route__date_to, gtfs_route__line_refs, gtfs_route_...

step 4:
 Thought: I need information
 Action: query_open_bus_api({"endpoint": "/gtfs_rides/list", "params

'מצטער, השאלה הזו דורשת עיבוד של כמות נתונים גדולה מדי עבור הגרסה החינמית הנוכחית. נסה לצמצם את טווח הזמן או לשאול על קו/תחנה ספציפיים.'

In [32]:
print(query_open_bus_api("/siri_ride_stops/list",
    '{"gtfs_route__route_short_name":"19",'
    '"siri_ride__scheduled_start_time_from":"2025-03-02T06:30:00Z",'
    '"siri_ride__scheduled_start_time_to":"2025-03-02T07:30:00Z",'
    '"order_by":"order asc",'
    '"limit":2}'))

Returned 2 record(s) (showing up to 50).
[{"id": 2470732327, "siri_stop_id": 8315, "siri_ride_id": 97431903, "order": 1, "gtfs_stop_id": null, "nearest_siri_vehicle_location_id": null, "siri_stop__code": 52104, "siri_ride__siri_route_id": 8058, "siri_ride__journey_ref": "2025-03-02-584932544", "siri_ride__scheduled_start_time": "2025-03-02T06:35:00+00:00", "siri_ride__vehicle_ref": "63552901", "siri_ride__updated_first_last_vehicle_locations": null, "siri_ride__first_vehicle_location_id": null, "siri_ride__last_vehicle_location_id": null, "siri_ride__updated_duration_minutes": null, "siri_ride__duration_minutes": null, "siri_ride__journey_gtfs_ride_id": null, "siri_ride__route_gtfs_ride_id": null, "siri_ride__gtfs_ride_id": null, "gtfs_stop__date": null, "gtfs_stop__code": null, "gtfs_stop__lat": null, "gtfs_stop__lon": null, "gtfs_stop__name": null, "gtfs_stop__city": null, "nearest_siri_vehicle_location__siri_snapshot_id": null, "nearest_siri_vehicle_location__siri_ride_stop_id": nul

## Streamlit

In [36]:
import re
from groq import BadRequestError, APIStatusError


def extract_coords(text: str):
    """Pull {lat, lon, label} triples (and bare lat/lon pairs) out of any text/JSON."""
    coords = []
    # JSON-ish objects with lat/lon (+optional label/name)
    for m in re.finditer(
        r'\{[^{}]*?"lat"\s*:\s*(-?\d+\.?\d*)[^{}]*?"lon"\s*:\s*(-?\d+\.?\d*)[^{}]*?\}', text):
        obj = m.group(0)
        lat, lon = float(m.group(1)), float(m.group(2))
        label_m = re.search(r'"(?:label|name|gtfs_stop__name|siri_stop__name)"\s*:\s*"([^"]+)"', obj)
        coords.append({"lat": lat, "lon": lon, "label": label_m.group(1) if label_m else ""})
    # de-dup
    seen, out = set(), []
    for c in coords:
        key = (round(c["lat"], 5), round(c["lon"], 5))
        if key not in seen and 29 < c["lat"] < 34 and 34 < c["lon"] < 36:  # Israel bbox sanity
            seen.add(key); out.append(c)
    return out


def react_agent_ui(question: str, max_steps: int = 15):
    """Like react_agent but RETURNS (final_answer, log_steps, coords) instead of printing."""
    messages = get_messages(question)
    log = []
    coords = []
    MAX_OBS_CHARS = 3000

    for step in range(max_steps):
        try:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools_schema,
                tool_choice="auto", parallel_tool_calls=False
            )
        except BadRequestError as e:
            if "tool_use_failed" in str(e):
                log.append({"type": "retry", "text": "tool_use_failed — מבקש מהמודל לנסות שוב"})
                messages.append({"role": "user", "content":
                    "Your previous tool call was malformed. Call exactly ONE tool using the "
                    "structured function-calling format, NOT as plain text. Try again."})
                continue
            raise
        except APIStatusError as e:
            if getattr(e, "status_code", None) == 413 or "rate_limit" in str(e) or "too large" in str(e).lower():
                final = ("מצטער, השאלה הזו דורשת עיבוד של כמות נתונים גדולה מדי עבור הגרסה החינמית "
                         "הנוכחית. נסה לצמצם את טווח הזמן או לשאול על קו/תחנה ספציפיים.")
                return final, log, coords
            raise

        msg = response.choices[0].message

        if not msg.tool_calls:
            coords += extract_coords(msg.content or "")
            return msg.content, log, coords

        messages.append(msg)
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_map[func_name](**args)

            log.append({"type": "action", "tool": func_name, "args": args,
                        "observation": result[:500]})
            coords += extract_coords(result)

            trimmed = result if len(result) <= MAX_OBS_CHARS else result[:MAX_OBS_CHARS] + "\n...[truncated]"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": trimmed})

    return "Max steps reached", log, coords

In [37]:
%%writefile app.py
import streamlit as st
import pandas as pd
import importlib.util, sys

# --- import the agent code from the notebook-exported module ---
# We re-create the agent objects by running the user's notebook cells via a helper module.
# Simplest path: paste your agent setup into agent_core.py (see tא 3 below) and import it.
from agent_core import react_agent_ui

st.set_page_config(page_title="Open Bus Agent", page_icon="🚌", layout="wide")
st.title("🚌 סוכן נתוני תחבורה ציבורית — Open Bus")
st.caption("שאל בשפה חופשית על תחנות, מספר נסיעות, מיקום חי ומהירות. הסוכן ניגש ל-Open Bus STRIDE API.")

EXAMPLES = [
    "מה תחנת המוצא של קו 189?",
    "כמה תחנות יש לקו 480?",
    "כמה נסיעות מתוכננות יש לקו 19 בין 07:00 ל-08:00 ב-1 בינואר 2023?",
    "הצג על מפה את רשימת התחנות של קו 18 לפי הסדר",
    "איפה נמצא כרגע אוטובוס של קו 5? (10 הדקות האחרונות)",
]

CANNOT = [
    "❌ איחור בפועל (מתוכנן מול אמת) — הנתון לרוב לא מחובר ולא אמין",
    "❌ מספר נוסעים / עומס — לא קיים ב-Open Bus",
    "❌ טווחי זמן ארוכים (יום/שבוע שלם) — חורג ממגבלת הטוקנים",
]

with st.sidebar:
    st.header("שאלות לדוגמה")
    for ex in EXAMPLES:
        if st.button(ex, use_container_width=True):
            st.session_state["q"] = ex
    st.divider()
    st.subheader("מה הסוכן לא יכול")
    for c in CANNOT:
        st.markdown(c)

q = st.text_input("שאלה:", value=st.session_state.get("q", ""), placeholder="לדוגמה: מה תחנת המוצא של קו 189?")
go = st.button("שאל את הסוכן", type="primary")

if go and q.strip():
    with st.spinner("הסוכן עובד..."):
        answer, log, coords = react_agent_ui(q)

    st.subheader("תשובה")
    st.write(answer)

    if coords:
        st.subheader("מפה")
        df = pd.DataFrame(coords)
        st.map(df[["lat", "lon"]])
        with st.expander("נקודות על המפה"):
            st.dataframe(df)

    with st.expander("🔍 שלבי הסוכן (ReAct log)"):
        for i, s in enumerate(log, 1):
            if s["type"] == "retry":
                st.markdown(f"**{i}.** ⟳ {s['text']}")
            else:
                st.markdown(f"**{i}. {s['tool']}**")
                st.code(str(s["args"]), language="json")
                st.text(s["observation"])

Writing app.py


In [38]:
%%writefile agent_core.py
import os, json, re, requests
from groq import Groq, BadRequestError, APIStatusError

# --- credentials & model ---
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")  # set below from Colab
MODEL = "llama-3.3-70b-versatile"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# --- PASTE HERE the bodies of: ---
#   OPEN_BUS_BASE_URL, HEADERS
#   get_open_bus_endpoints(...)
#   query_open_bus_api(...)
#   tools_map
#   tools_schema
#   system_prompt
#   get_messages(...)
#   extract_coords(...)
#   react_agent_ui(...)
# (copy them verbatim from your notebook cells)

Writing agent_core.py


In [41]:
# התקנות (פעם אחת)
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# ודא שמפתח ה-API בסביבה של התהליך
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# הפעל את Streamlit ברקע
!streamlit run app.py &>/content/st_log.txt &
!sleep 5

# פתח מנהרת cloudflare — ה-URL יודפס בלוג
!nohup ./cloudflared tunnel --url http://localhost:8501 &>/content/cf_log.txt &
!sleep 8
!grep -o 'https://[-a-z0-9]*\.trycloudflare\.com' /content/cf_log.txt | head -1

https://turner-mls-sql-synopsis.trycloudflare.com


In [40]:
!curl -s https://loca.lt/mytunnelpassword

104.196.166.100